# Development of the MLP architecture

## Imports

In [5]:
from protgen.data import ProteinDataset
from torch.utils.data import DataLoader
from protgen.data import AA, FAMILIES, IGNORE_INDEX, special_tokens
import torch
import torch.nn.functional as F

## Data

In [6]:
from protgen.data import ProteinDataset
from torch.utils.data import DataLoader

train_data = ProteinDataset(path_to_data="../data/train.csv")
val_data = ProteinDataset(path_to_data="../data/val.csv")
test_data = ProteinDataset(path_to_data="../data/test.csv")

print(f"Data class: {train_data}")

# Get tensors
X_train = train_data.tokens
y_train_labels = train_data.labels          # classification task
y_train_shifted = train_data.tokens_shifted # generative task

print(f"# Train data: \n Tokens shape: {X_train.shape}\n Labels (shape): {y_train_labels.shape} \n Shifted shape: {y_train_shifted.shape}")


# Inddexing
x, x_shifted, y, length = train_data[0]
print(f"# Sample 0: \nTokens: \n {x}\n Shifted: \n {x_shifted}\n Label: {y}\n Length: {length}")

# DataLoader
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

Data class: ProteinDataset(num_samples=16020, vocab_size=23, num_classes=3, max_length=129)
# Train data: 
 Tokens shape: torch.Size([16020, 129])
 Labels (shape): torch.Size([16020]) 
 Shifted shape: torch.Size([16020, 129])
# Sample 0: 
Tokens: 
 tensor([ 1, 13, 10, 11, 11, 16,  7, 14, 10, 10, 11, 14, 19,  8,  7,  3, 17, 15,
         3, 19, 12, 12, 20, 11, 12, 18, 16, 17,  7, 19, 18,  5, 10, 18, 12,  6,
        22, 16,  8, 11, 19, 10, 14, 12, 11,  9, 18,  3, 11, 18, 10, 13,  5, 10,
        13, 18, 12, 18, 10, 17, 15,  8, 19, 11,  7, 22, 10, 11,  3,  6,  8, 10,
         5,  6, 20, 16,  3, 10, 16, 12, 10, 16,  5, 22, 12, 22, 11,  6, 11, 12,
        10, 14, 22,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0])
 Shifted: 
 tensor([13, 10, 11, 11, 16,  7, 14, 10, 10, 11, 14, 19,  8,  7,  3, 17, 15,  3,
        19, 12, 12, 20, 11, 12, 18, 16, 17,  7, 19, 18,  5, 10, 18, 12,  6, 22

In [7]:
train_data 

ProteinDataset(num_samples=16020, vocab_size=23, num_classes=3, max_length=129)

In [8]:
train_data.vocab

{'<PAD>': 0,
 '<BOS>': 1,
 '<EOS>': 2,
 'A': 3,
 'C': 4,
 'D': 5,
 'E': 6,
 'F': 7,
 'G': 8,
 'H': 9,
 'I': 10,
 'K': 11,
 'L': 12,
 'M': 13,
 'N': 14,
 'P': 15,
 'Q': 16,
 'R': 17,
 'S': 18,
 'T': 19,
 'V': 20,
 'W': 21,
 'Y': 22}

In [11]:
train_data.df

,sequence,family
0,MIKKQFNIIKNTGFARPATLLVKLSQRFTSDISLEYQGKTINLKHS...,HPr
1,MNKHELITRISEKSGMTKKDVESVLNGFMDEVTDALGSGNKVQLIG...,HU
2,MTQRFEDTFLIQNKLGLHARAASVLAELSAQFDANITLFQGSKEAE...,HPr
3,MNKTELILRIAEETGFSRKNVELGLNAAINLITKALTEEEKVQIVG...,HU
4,MVERKAVIKNEAGIHCRPTAVITQEAAKVDAVIMILSPGGTCQLGS...,HPr
...,...,...
16015,MTKADIINEIANETGVAKKDVTVVVESFMATIKDSLLEKRECVYLR...,HU
16016,MSRLSRQVTIVNQKGLHARASAKFVTFVSRLPEGLKVEVEKDGQCV...,HPr
16017,MNKSDLINHVAAETGLTKADSGFALEAVLEGITKSLRKGDTVTLTG...,HU
16018,MTLTKAELADLLFEKVGLNKREAKDMVEAFFEEIRNALERGEGVKL...,HU


In [14]:
129//4

32

In [16]:
train_data.tokens

tensor([[ 1, 13, 10,  ...,  0,  0,  0],
        [ 1, 13, 14,  ...,  0,  0,  0],
        [ 1, 13, 19,  ...,  0,  0,  0],
        ...,
        [ 1, 13, 14,  ...,  0,  0,  0],
        [ 1, 13, 19,  ...,  0,  0,  0],
        [ 1, 13,  3,  ...,  0,  0,  0]])

In [ ]:
class ProteinDataset_MLP(ProteinDataset):

    def __init__(self, path_to_data):

        # Set token and family mappings
        self.vocab = dict(zip(total_tokens, range(len(total_tokens))))
        self.fam_to_idx = dict(zip(FAMILIES, range(len(FAMILIES))))
        self.window_size_mlp = 3

        # Load raw CSV files
        self.df = pd.read_csv(path_to_data)
        self.max_length = 127 + 2    #Longest sequence + <BOS> / <EOS>

        self.tokens, self.tokens_shifted, self.labels, self.lengths = [], [], [], []
        # encode proteins and familiy
        for sequence, label in self.df[["sequence", "family"]].values:
            tokens = self.encode(sequence)
            length = len(sequence)
            shifted_tokens = self.shift_tokens(tokens, length)

            # Create the MLP specfic tokens and shifted_tokens
            # I decided to do a window with stride 3, we can do a different aproach of course
            for index_mlp_token in range(self.max_length//self.window_size_mlp):
                
                # Append to lists
                self.tokens.append(tokens[index_mlp_token*3:(index_mlp_token+1)*3])
                self.tokens_shifted.append(shifted_tokens[index_mlp_token*3])
                self.labels.append(self.fam_to_idx[label])
                self.lengths.append(length)

        self.tokens = torch.stack(self.tokens)
        self.tokens_shifted = torch.stack(self.tokens_shifted)
        self.labels = torch.tensor(self.labels, dtype=torch.long)

## Model

**Pseudo Code**

Torch stuff which I do not remember but I will check on the classes

1. Prepare our data to do this (I do not belive that we have this now) We need to break it into 3 size proteins.
    - The question that I have now is do we do this inside or outside of our loop? (I did not found any existant torch
    structure that handles this automatically)
    - Should we use all the data just consider windows of 3


2. We need to have a lookup table with the same dimension as our vocab. In each iteration we go search that one (this needs to be inside the torch loop and the lookup table needs to be optimizable). Number of dimensions in here is optimizable as well)

3. Now we concatenate them! Let's do this yee

4. Architecture to define in a standart normal neural network and the label to predict is the label at the end.
    - They suggest using tanh

5. At the end a softmax because the output should be all possible tokens

## Questions

- How do we deal with the beggining in the MLP if we want to generate from it after and we always need 3 residues to start. Is the solution something like this "like this  [ 1, 1, 1] -> 13 we should"

- 